# Laboratorio 06 — Agente con tools: web search + calculadora + base de datos
### Python AI Developer 2026 · Capítulo 4: Agentes de IA

**Objetivo:** construir un agente ReAct que decide autónomamente qué herramienta usar para responder cada pregunta.

**La diferencia con RAG:** en RAG el pipeline es fijo — siempre busca en documentos. Un agente decide en cada paso qué hacer: buscar en la web, calcular, consultar una base de datos, o responder directamente.

**Duración:** 90 min  
**Setup:**
```bash
uv add langgraph langchain-anthropic langchain-core langchain-community tavily-python pandas python-dotenv
```

---
## Parte 1 — El patrón ReAct

ReAct (Reasoning + Acting) es el patrón detrás de todos los agentes modernos. El modelo alterna entre:

```
Thought:     "El usuario quiere saber el precio actual del dólar. Necesito buscarlo en la web."
Action:      buscar_web("tipo de cambio dólar soles hoy")
Observation: "El dólar cotiza a S/. 3.72 según el BCRP..."
Thought:     "Ya tengo la información. Puedo responder."
Answer:      "El tipo de cambio hoy es S/. 3.72"
```

El ciclo se repite hasta que el agente considera que tiene suficiente información para responder.

In [ ]:
# --- imports ---
# para tavily ir a https://tavily.com/ y crear cuenta gratuita
import os
import sqlite3
import pandas as pd
from dotenv import load_dotenv

from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

load_dotenv()

# verificar credenciales necesarias
assert os.getenv("ANTHROPIC_API_KEY"), "Falta ANTHROPIC_API_KEY en .env"
assert os.getenv("TAVILY_API_KEY"),    "Falta TAVILY_API_KEY en .env — registrarse en tavily.com"

print("Credenciales OK")

Credenciales OK


---
## Parte 2 — Crear la base de datos SQLite

Usamos SQLite porque no requiere servidor — es un archivo local. Simulamos una base de datos de ventas de una empresa de alimentos.

In [3]:
# crear base de datos SQLite con datos de ventas ficticios
conn = sqlite3.connect("ventas.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS ventas")
cursor.execute("""
    CREATE TABLE ventas (
        id INTEGER PRIMARY KEY,
        producto TEXT,
        categoria TEXT,
        cliente TEXT,
        region TEXT,
        mes TEXT,
        unidades INTEGER,
        monto_soles REAL
    )
""")

datos = [
    (1,  "Harina 50kg",     "harinas",  "Wong",         "Lima",       "2025-01", 320,  45600.0),
    (2,  "Harina 50kg",     "harinas",  "Metro",        "Lima",       "2025-01", 280,  39900.0),
    (3,  "Fideos 500g",     "pastas",   "Wong",         "Lima",       "2025-01", 1500, 12000.0),
    (4,  "Fideos 500g",     "pastas",   "Makro",        "Arequipa",   "2025-01", 900,  7200.0),
    (5,  "Aceite 1L",       "aceites",  "Plaza Vea",   "Lima",       "2025-01", 600,  18000.0),
    (6,  "Harina 50kg",     "harinas",  "Wong",         "Lima",       "2025-02", 350,  49875.0),
    (7,  "Harina 50kg",     "harinas",  "Makro",        "Trujillo",   "2025-02", 200,  28500.0),
    (8,  "Fideos 500g",     "pastas",   "Metro",        "Lima",       "2025-02", 1200, 9600.0),
    (9,  "Aceite 1L",       "aceites",  "Wong",         "Lima",       "2025-02", 750,  22500.0),
    (10, "Galletas 200g",   "snacks",   "Plaza Vea",   "Arequipa",   "2025-02", 2000, 14000.0),
    (11, "Harina 50kg",     "harinas",  "Metro",        "Lima",       "2025-03", 410,  58425.0),
    (12, "Fideos 500g",     "pastas",   "Wong",         "Trujillo",   "2025-03", 800,  6400.0),
    (13, "Aceite 1L",       "aceites",  "Makro",        "Lima",       "2025-03", 500,  15000.0),
    (14, "Galletas 200g",   "snacks",   "Metro",        "Lima",       "2025-03", 1800, 12600.0),
    (15, "Harina 50kg",     "harinas",  "Wong",         "Arequipa",   "2025-03", 180,  25650.0),
]
cursor.executemany("INSERT INTO ventas VALUES (?,?,?,?,?,?,?,?)", datos)
conn.commit()
conn.close()

# verificar
df = pd.read_sql("SELECT * FROM ventas", sqlite3.connect("ventas.db"))
print(f"Base de datos creada: {len(df)} registros")
print(df.head(3).to_string())

Base de datos creada: 15 registros
   id     producto categoria cliente region      mes  unidades  monto_soles
0   1  Harina 50kg   harinas    Wong   Lima  2025-01       320      45600.0
1   2  Harina 50kg   harinas   Metro   Lima  2025-01       280      39900.0
2   3  Fideos 500g    pastas    Wong   Lima  2025-01      1500      12000.0


---
## Parte 3 — Definir las herramientas del agente

Las herramientas son funciones Python decoradas con `@tool`. LangChain usa el docstring para que el agente entienda para qué sirve cada herramienta y cuándo usarla. El docstring es crítico — es el "manual" que lee el modelo.

In [ ]:
@tool
def calcular(expresion: str) -> str:
    """
    Evalúa una expresión matemática y devuelve el resultado.
    Usar para cálculos aritméticos: sumas, porcentajes, promedios, variaciones.
    Ejemplos de expresión: '1500 * 0.18', '(450 - 380) / 380 * 100', '(120 + 95 + 88) / 3'
    """
    try:
        # eval limitado a operaciones matemáticas básicas
        resultado = eval(expresion, {"__builtins__": {}}, {})
        return f"{resultado:.4f}"
    except Exception as e:
        return f"Error al calcular '{expresion}': {e}"

@tool
def consultar_bd(query_sql: str) -> str:
    """
    Ejecuta una consulta SQL sobre la base de datos de ventas y devuelve los resultados.
    La tabla se llama 'ventas' y tiene las columnas:
    id, producto, categoria, cliente, region, mes, unidades, monto_soles.
    Usar para preguntas sobre ventas históricas, rankings, totales por región o producto.
    Solo se permiten consultas SELECT.
    """
    # solo permitir SELECT para seguridad
    if not query_sql.strip().upper().startswith("SELECT"):
        return "Solo se permiten consultas SELECT."
    try:
        conn = sqlite3.connect("ventas.db")
        df = pd.read_sql(query_sql, conn)
        conn.close()
        if df.empty:
            return "La consulta no devolvió resultados."
        return df.to_string(index=False)
    except Exception as e:
        return f"Error en la consulta SQL: {e}"


# herramienta de búsqueda web usando Tavily
buscar_web = TavilySearchResults(
    max_results=3,
    description="Busca información actual en internet. Usar para preguntas sobre precios de mercado, noticias, tasas de cambio o cualquier información que no esté en la base de datos.",
)

tools = [calcular, consultar_bd, buscar_web]
print(f"Herramientas definidas: {[t.name for t in tools]}")

Herramientas definidas: ['calcular', 'consultar_bd', 'tavily_search_results_json']


C:\Users\djace\AppData\Local\Temp\ipykernel_14064\512313907.py:40: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  buscar_web = TavilySearchResults(


---
## Parte 4 — Crear el agente ReAct

In [5]:
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0,  # temperatura 0 para que el agente sea determinista
    max_tokens=1000,
)

# create_react_agent arma el ciclo Thought-Action-Observation automáticamente
# solo necesitamos pasarle el LLM y las herramientas disponibles
agente = create_react_agent(llm, tools)

print("Agente ReAct creado")
print(f"Herramientas disponibles: {[t.name for t in tools]}")

Agente ReAct creado
Herramientas disponibles: ['calcular', 'consultar_bd', 'tavily_search_results_json']


C:\Users\djace\AppData\Local\Temp\ipykernel_14064\2863341189.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(llm, tools)


---
## Parte 5 — Función auxiliar para ejecutar el agente y ver su razonamiento

In [6]:
def ejecutar_agente(pregunta: str) -> str:
    """
    Ejecuta el agente con una pregunta y muestra el razonamiento paso a paso.
    Retorna la respuesta final.
    """
    print(f"Pregunta: {pregunta}")
    print("-" * 50)

    resultado = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})

    # los mensajes incluyen el razonamiento interno del agente
    for msg in resultado["messages"]:
        tipo = type(msg).__name__
        if tipo == "AIMessage":
            # si tiene tool_calls, el agente está eligiendo una herramienta
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"[Herramienta elegida] {tc['name']}")
                    print(f"[Argumentos] {tc['args']}")
            else:
                # si no tiene tool_calls, es la respuesta final
                print(f"[Respuesta] {msg.content}")
        elif tipo == "ToolMessage":
            # resultado de ejecutar la herramienta
            print(f"[Resultado herramienta] {str(msg.content)[:200]}")

    print()
    # la respuesta final es el último AIMessage sin tool_calls
    for msg in reversed(resultado["messages"]):
        if type(msg).__name__ == "AIMessage" and not msg.tool_calls:
            return msg.content
    return ""

---
## Parte 6 — Pruebas: el agente elige la herramienta correcta

El momento pedagógico clave: el agente usa herramientas distintas para preguntas distintas, sin que le digamos cuál usar.

In [ ]:
# Pregunta 1: requiere consultar la base de datos
ejecutar_agente("¿Cuál fue el producto con mayor monto de ventas en enero de 2025?")

Pregunta: ¿Cuál fue el producto con mayor monto de ventas en enero de 2025?
--------------------------------------------------
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': "SELECT producto, SUM(monto_soles) as total_ventas FROM ventas WHERE mes = 'enero' AND YEAR(fecha) = 2025 GROUP BY producto ORDER BY total_ventas DESC LIMIT 1"}
[Resultado herramienta] Error en la consulta SQL: Execution failed on sql 'SELECT producto, SUM(monto_soles) as total_ventas FROM ventas WHERE mes = 'enero' AND YEAR(fecha) = 2025 GROUP BY producto ORDER BY total_ventas DESC
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': "SELECT producto, SUM(monto_soles) as total_ventas FROM ventas WHERE mes = 'enero' GROUP BY producto ORDER BY total_ventas DESC LIMIT 1"}
[Resultado herramienta] La consulta no devolvió resultados.
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': 'SELECT DISTINCT mes FROM ventas LIMIT 10'}
[Resultado herramienta]     mes
2025-01
2025-02
2025-03
[He

In [ ]:
# Pregunta 2: requiere consultar BD y luego calcular
ejecutar_agente(
    "¿Cuánto creció el monto total de ventas de Harina 50kg entre enero y febrero de 2025? "
    "Dame el porcentaje de crecimiento."
)

Pregunta: ¿Cuánto creció el monto total de ventas de Harina 50kg entre enero y febrero de 2025? Dame el porcentaje de crecimiento.
--------------------------------------------------
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': "SELECT mes, SUM(monto_soles) as total_ventas FROM ventas WHERE producto = 'Harina 50kg' AND mes IN ('enero', 'febrero') GROUP BY mes ORDER BY mes"}
[Resultado herramienta] La consulta no devolvió resultados.
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': "SELECT DISTINCT producto FROM ventas WHERE producto LIKE '%Harina%' LIMIT 10"}
[Resultado herramienta]    producto
Harina 50kg
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': "SELECT DISTINCT mes FROM ventas WHERE producto = 'Harina 50kg' ORDER BY mes"}
[Resultado herramienta]     mes
2025-01
2025-02
2025-03
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': "SELECT mes, SUM(monto_soles) as total_ventas FROM ventas WHERE producto = 'Harina 50kg' AND mes I

In [11]:
# Pregunta 3: requiere búsqueda web
ejecutar_agente("¿Cuál es el precio actual del trigo en los mercados internacionales?")

Pregunta: ¿Cuál es el precio actual del trigo en los mercados internacionales?
--------------------------------------------------
[Herramienta elegida] tavily_search_results_json
[Argumentos] {'query': 'precio actual trigo mercados internacionales'}
[Resultado herramienta] [{"title": "Mercados Mundiales hoy (1 de marzo de 2026): precio del trigo en ...", "url": "https://www.agronewscastillayleon.com/mercados-mundiales-agrarios", "content": "Estos son los precios oficial
[Respuesta] Basándome en la información actual de los mercados internacionales, aquí está el precio del trigo:

## **Precios Actuales del Trigo (Marzo 2026)**

### **Principales Cotizaciones:**

- **París (Euronext):** **197,5 €/tonelada** - en alza
- **Chicago (CBOT):** **183,84 €/tonelada** - alcanza su nivel más alto en un año

### **Contratos de Futuros:**
- **Trigo Septiembre 2026:** 200,75 €/t
- **Trigo Diciembre 2026:** 208,75 €/t

### **Factores que Influyen en los Precios:**

1. **Tensión Geopolítica:** La situ

'Basándome en la información actual de los mercados internacionales, aquí está el precio del trigo:\n\n## **Precios Actuales del Trigo (Marzo 2026)**\n\n### **Principales Cotizaciones:**\n\n- **París (Euronext):** **197,5 €/tonelada** - en alza\n- **Chicago (CBOT):** **183,84 €/tonelada** - alcanza su nivel más alto en un año\n\n### **Contratos de Futuros:**\n- **Trigo Septiembre 2026:** 200,75 €/t\n- **Trigo Diciembre 2026:** 208,75 €/t\n\n### **Factores que Influyen en los Precios:**\n\n1. **Tensión Geopolítica:** La situación entre Irán y Estados Unidos está impulsando al alza los precios agrarios, especialmente en Chicago\n\n2. **Volatilidad del Mercado:** El mercado del trigo es altamente volátil, con fluctuaciones diarias significativas\n\n3. **Tipo de Cambio:** El dólar juega un papel crucial, ya que el trigo se cotiza en dólares en los mercados internacionales\n\n4. **Condiciones Climáticas:** Las lluvias y perspectivas de cosecha afectan las cotizaciones\n\n5. **Presión de Rus

In [12]:
# Pregunta 4: requiere BD + web + cálculo (multi-step)
ejecutar_agente(
    "Las ventas totales de aceite en la BD están en soles. "
    "¿A cuánto equivalen en dólares al tipo de cambio actual?"
)

Pregunta: Las ventas totales de aceite en la BD están en soles. ¿A cuánto equivalen en dólares al tipo de cambio actual?
--------------------------------------------------
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': "SELECT SUM(monto_soles) as total_soles FROM ventas WHERE producto = 'aceite'"}
[Herramienta elegida] tavily_search_results_json
[Argumentos] {'query': 'tipo de cambio soles a dólares actual'}
[Resultado herramienta] total_soles
       None
[Resultado herramienta] [{"title": "Cotización USD/PEN - Cambio Dólar Sol peruano hoy - Investing.com", "url": "https://es.investing.com/currencies/usd-pen", "content": "Consulte el tipo de cambio de Dólar a Sol peruano y la
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': 'SELECT DISTINCT producto FROM ventas'}
[Resultado herramienta]      producto
  Harina 50kg
  Fideos 500g
    Aceite 1L
Galletas 200g
[Herramienta elegida] consultar_bd
[Argumentos] {'query_sql': "SELECT SUM(monto_soles) as total_soles FR

'## Resultado\n\nLas ventas totales de **Aceite 1L** en la base de datos son:\n\n- **55,500 soles peruanos**\n\nAl tipo de cambio actual de **1 USD = 3.4682 PEN**, esto equivale a:\n\n- **$16,002.54 USD** (dólares estadounidenses)'

In [13]:
# Pregunta 5: el agente responde directamente sin herramientas
ejecutar_agente("¿Qué es el índice de Gini y para qué se usa?")

Pregunta: ¿Qué es el índice de Gini y para qué se usa?
--------------------------------------------------
[Respuesta] # Índice de Gini

El **Índice de Gini** es una medida estadística que cuantifica la desigualdad en la distribución de una variable, siendo más comúnmente utilizado para medir la **desigualdad de ingresos o riqueza** en una población.

## Características principales:

**Rango de valores:**
- Varía entre **0 y 1** (o 0% a 100%)
- **0** = Igualdad perfecta (todos tienen los mismos ingresos)
- **1** = Desigualdad máxima (una persona tiene todo, el resto nada)

## Usos principales:

1. **Análisis económico y social**
   - Medir la desigualdad de ingresos en países
   - Comparar desigualdad entre diferentes regiones o períodos

2. **Política pública**
   - Evaluar el impacto de políticas redistributivas
   - Diseñar programas de reducción de pobreza

3. **Investigación académica**
   - Estudios sobre desarrollo económico
   - Análisis de equidad social

4. **Análisis de datos

'# Índice de Gini\n\nEl **Índice de Gini** es una medida estadística que cuantifica la desigualdad en la distribución de una variable, siendo más comúnmente utilizado para medir la **desigualdad de ingresos o riqueza** en una población.\n\n## Características principales:\n\n**Rango de valores:**\n- Varía entre **0 y 1** (o 0% a 100%)\n- **0** = Igualdad perfecta (todos tienen los mismos ingresos)\n- **1** = Desigualdad máxima (una persona tiene todo, el resto nada)\n\n## Usos principales:\n\n1. **Análisis económico y social**\n   - Medir la desigualdad de ingresos en países\n   - Comparar desigualdad entre diferentes regiones o períodos\n\n2. **Política pública**\n   - Evaluar el impacto de políticas redistributivas\n   - Diseñar programas de reducción de pobreza\n\n3. **Investigación académica**\n   - Estudios sobre desarrollo económico\n   - Análisis de equidad social\n\n4. **Análisis de datos empresariales**\n   - Distribución de ventas entre clientes\n   - Concentración de ingresos

---
## Parte 7 — Análisis del comportamiento del agente

Observa el patrón en las respuestas anteriores: el agente nunca elige una herramienta al azar. El docstring de cada herramienta es lo que guía esa decisión.

In [14]:
# resumen de qué herramientas se usaron en cada caso
casos = [
    ("¿Cuál fue el producto top en enero?",              "consultar_bd"),
    ("¿Cuánto creció Harina entre enero y febrero?",     "consultar_bd → calcular"),
    ("¿Precio actual del trigo internacional?",           "buscar_web"),
    ("¿Ventas de aceite en dólares al tipo de cambio?",  "consultar_bd → buscar_web → calcular"),
    ("¿Qué es el índice de Gini?",                        "ninguna (conocimiento propio)"),
]

print(f"{'Pregunta':<50} {'Herramientas usadas'}")
print("-" * 80)
for pregunta, herramientas in casos:
    print(f"{pregunta:<50} {herramientas}")

print()
print("El agente combina herramientas en secuencia cuando ninguna sola es suficiente.")
print("Eso es lo que diferencia un agente de un pipeline fijo como RAG.")

Pregunta                                           Herramientas usadas
--------------------------------------------------------------------------------
¿Cuál fue el producto top en enero?                consultar_bd
¿Cuánto creció Harina entre enero y febrero?       consultar_bd → calcular
¿Precio actual del trigo internacional?            buscar_web
¿Ventas de aceite en dólares al tipo de cambio?    consultar_bd → buscar_web → calcular
¿Qué es el índice de Gini?                         ninguna (conocimiento propio)

El agente combina herramientas en secuencia cuando ninguna sola es suficiente.
Eso es lo que diferencia un agente de un pipeline fijo como RAG.
